In [17]:
import gc
import os
import sys
import warnings

import numpy as np
import optuna
import pandas as pd
import torch

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler
from metrics import compute_metrics_stats

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
name = "nci"
PATH = f"../{name}_data/"
method = "GATv2"

In [4]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data(name)

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00, 10.36it/s]


Done!


In [10]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_float("dropout1", 0.1, 0.5, step=0.05),
        "dropout2": trial.suggest_float("dropout2", 0.1, 0.5, step=0.05),
        "hidden1": trial.suggest_int("hidden1", 256, 1024),
        "hidden2": trial.suggest_int("hidden2", 64, min(512, trial.params["hidden1"])),
        "hidden3": trial.suggest_int("hidden3", 32, min(256, trial.params["hidden2"])),
        "epochs": trial.suggest_int("epochs", 100, 10000, step=100),
        "heads": trial.suggest_int("heads", 2, 8),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical("scheduler", [None, "Cosine"]),
        "gnn_layer": method,
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        # T_maxの最小値を1以上に保証
        min_epoch_div = max(1, params["epochs"] // 5)  # 最小値1を強制
        max_epoch_div = max(
            min_epoch_div + 1, params["epochs"] // 2
        )  # low < highを保証

        params["T_max"] = trial.suggest_int(
            "T_max", low=min_epoch_div, high=max_epoch_div
        )

        # 追加のチェック（防御的プログラミング）
        if params["T_max"] <= 0:
            raise optuna.TrialPruned(f"Invalid T_max: {params['T_max']}")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()

        metrics_collector = {"acc": [], "f1": [], "auroc": [], "aupr": []}

        true_datas = pd.DataFrame()
        predict_datas = pd.DataFrame()

        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = (
                drGAT.train(sampler, params=params, device=device, verbose=False)
            )

            true_datas = pd.concat(
                [true_datas, pd.DataFrame(best_val_labels)],
                ignore_index=True,
                axis=1,
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(best_val_prob)],
                ignore_index=True,
                axis=1,
            )

            del sampler, best_val_labels, best_val_prob
            torch.cuda.empty_cache()
            gc.collect()

        true_datas = true_datas.T
        predict_datas = predict_datas.T

        metrics_result = compute_metrics_stats(
            trial=trial,
            true=true_datas,
            pred=predict_datas,
            target_metrics=["AUROC", "AUPR", "F1", "ACC"],
        )

        return tuple(metrics_result["target_values"])

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            raise optuna.TrialPruned(f"OOM at trial {trial.number}")
        else:
            print(f"RuntimeError in trial {trial.number}: {str(e)}")
            raise e

    except ZeroDivisionError:
        print(f"Pruned trial {trial.number}: ZeroDivisionError in CosineAnnealingLR")
        raise optuna.TrialPruned("ZeroDivisionError in CosineAnnealingLR")
    except Exception as e:
        print(f"Unexpected error in trial {trial.number}: {str(e)}")
        raise e

In [15]:
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.NSGAIISampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage=f"sqlite:///{method}.sqlite3",
    study_name=method,
    load_if_exists=True,
)
study.optimize(objective, n_trials=1000)

[I 2025-04-03 12:57:31,592] Using an existing study with name 'GATv2' instead of creating a new one.


Using device: cuda


  0%|          | 0/3 [00:00<?, ?it/s]
[I 2025-04-03 12:57:33,735] Trial 71 pruned. OOM at trial 71


Pruned trial 71: CUDA OOM
Using device: cuda


  0%|          | 0/3 [00:00<?, ?it/s]

...

Pruned trial 102: CUDA OOM



[I 2025-04-03 12:59:03,884] Trial 102 pruned. OOM at trial 102


Using device: cuda


  0%|          | 0/3 [00:00<?, ?it/s]
[W 2025-04-03 12:59:05,849] Trial 103 failed with parameters: {'dropout1': 0.45000000000000007, 'dropout2': 0.30000000000000004, 'hidden1': 919, 'hidden2': 304, 'hidden3': 65, 'heads': 8, 'activation': 'gelu', 'optimizer': 'AdamW', 'lr': 0.0008627055697713341, 'weight_decay': 0.002017406287954001, 'scheduler': 'Cosine', 'T_max': 2} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/nn/conv/message_passing.py", line 317, in _index_select_safe
    return src.index_select(self.node_dim, index)
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 24.41 GiB. GPU 0 has a total capacity of 79.15 GiB of which 5.89 GiB is free. Including non-PyTorch memory, this process has 73.25 GiB memory in use. Of the allocated memory 34.11 GiB is allocated by PyTorch, and 38.64 GiB is reserved by PyTorch but unallocated. If reserved but unal

KeyboardInterrupt: 

In [16]:
study.trials_dataframe().dropna()

,number,values_0,values_1,values_2,values_3,datetime_start,datetime_complete,duration,params_T_max,params_activation,...,user_attrs_MCC_mean,user_attrs_MCC_std,user_attrs_Precision_mean,user_attrs_Precision_std,user_attrs_Recall_mean,user_attrs_Recall_std,user_attrs_Specificity_mean,user_attrs_Specificity_std,system_attrs_nsga2:generation,state
26,26,0.463196,0.475763,0.414121,0.499833,2025-04-03 12:54:58.879806,2025-04-03 12:55:16.386229,0 days 00:00:17.506423,2.0,gelu,...,-0.000864,0.001932,0.397852,0.222455,0.607610,0.537471,0.392056,0.537029,0,COMPLETE
49,49,0.484587,0.493241,0.400000,0.500000,2025-04-03 12:56:02.881161,2025-04-03 12:56:21.032102,0 days 00:00:18.150941,1.0,relu,...,0.000000,0.000000,0.300000,0.273861,0.600000,0.547723,0.400000,0.547723,0,COMPLETE
77,77,0.545149,0.540299,0.257441,0.494023,2025-04-03 12:57:53.559830,2025-04-03 12:58:13.347981,0 days 00:00:19.788151,2.0,relu,...,-0.017758,0.045286,0.196665,0.269360,0.373306,0.513344,0.614739,0.528177,0,COMPLETE


In [19]:
study = optuna.create_study(
    storage=f"sqlite:///{method}.sqlite3",
    study_name=method,
    load_if_exists=True,
)

[I 2025-04-03 13:17:11,653] Using an existing study with name 'GATv2' instead of creating a new one.


In [21]:
study.trials_dataframe()

,number,values_0,values_1,values_2,values_3,datetime_start,datetime_complete,duration,params_T_max,params_activation,...,params_heads,params_hidden1,params_hidden2,params_hidden3,params_lr,params_optimizer,params_scheduler,params_weight_decay,system_attrs_nsga2:generation,state
0,0,None,None,None,None,2025-04-03 13:05:53.530528,2025-04-03 13:05:58.334060,0 days 00:00:04.803532,NaN,relu,...,6,444,415,224,0.000018,AdamW,None,0.000825,0,PRUNED
1,1,None,None,None,None,2025-04-03 13:05:58.360321,2025-04-03 13:06:00.133754,0 days 00:00:01.773433,2114.0,gelu,...,6,693,123,63,0.000162,AdamW,Cosine,0.001684,0,PRUNED
2,2,None,None,None,None,2025-04-03 13:06:00.161771,2025-04-03 13:06:02.016073,0 days 00:00:01.854302,NaN,relu,...,4,695,284,225,0.004533,AdamW,None,0.002796,0,PRUNED
3,3,None,None,None,None,2025-04-03 13:06:02.041983,2025-04-03 13:06:03.657018,0 days 00:00:01.615035,NaN,gelu,...,6,997,122,80,0.000069,AdamW,None,0.000026,0,PRUNED
4,4,None,None,None,None,2025-04-03 13:06:03.682677,2025-04-03 13:06:06.928857,0 days 00:00:03.246180,900.0,relu,...,3,812,220,69,0.003156,AdamW,Cosine,0.000002,0,PRUNED
5,5,None,None,None,None,2025-04-03 13:06:06.955544,2025-04-03 13:06:08.856753,0 days 00:00:01.901209,2514.0,relu,...,8,931,303,233,0.003594,AdamW,Cosine,0.000010,0,PRUNED
6,6,None,None,None,None,2025-04-03 13:06:08.882612,2025-04-03 13:06:10.451464,0 days 00:00:01.568852,NaN,relu,...,5,894,458,137,0.000091,Adam,None,0.000010,0,PRUNED
7,7,None,None,None,None,2025-04-03 13:06:10.478827,2025-04-03 13:06:11.988209,0 days 00:00:01.509382,1621.0,gelu,...,7,276,87,71,0.000350,AdamW,Cosine,0.000022,0,PRUNED
8,8,None,None,None,None,2025-04-03 13:06:12.013820,2025-04-03 13:06:13.719865,0 days 00:00:01.706045,286.0,gelu,...,2,837,441,159,0.000134,Adam,Cosine,0.000001,0,PRUNED
9,9,None,None,None,None,2025-04-03 13:06:13.746431,2025-04-03 13:06:15.423845,0 days 00:00:01.677414,3512.0,relu,...,8,560,498,199,0.002025,AdamW,Cosine,0.000004,0,PRUNED
